# Build Joint Dataset: Data Gathering, Join & Save

This notebook **fetches and joins** all sources into a single county–year dataset and **saves** it to CSV. Exploratory analysis is in **`joint_eda.ipynb`** (loads the saved file).

**Steps:**
1. Load USGS Pesticide (county–year)
2. Load Census ACS + NCHS urban–rural (county-level)
3. Load USDA Cropland + build county_crop_year_ag
4. Optional experimental bridge (archived in ipm-testing)
5. Load CDC PLACES (2018 and 2019)
6. Join all on FIPS (+ YEAR for PLACES and county_year_collapsed)
7. **Save** → `data/joint_county_year_2018_2019.csv`

**Target years:** 2018 and 2019. **Join key:** 5-digit FIPS; PLACES and county_year_collapsed use FIPS + YEAR.

**Requirements:** `pandas`, `numpy`, `requests`. Optional: `rpy2` + R + CDCPLACES for PLACES.


In [ ]:
import gc
import os
import re
import time
import warnings
from io import StringIO
from concurrent.futures import ThreadPoolExecutor, as_completed
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import requests
import tempfile

# Optional: matplotlib, seaborn, geopandas (only needed for inline plots)
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    plt = sns = None
try:
    import geopandas as gpd
except ImportError:
    gpd = None

# Optional: rpy2 for CDC PLACES (if missing, PLACES cell will use fallback)
try:
    from rpy2.robjects import r
    from rpy2.robjects.packages import importr
    from rpy2.robjects.vectors import StrVector
    HAS_RPY2 = True
except ImportError:
    HAS_RPY2 = False

warnings.filterwarnings("ignore", category=FutureWarning)

# --- Stability / light-run config (set LIGHT_RUN=True if notebook crashes or runs out of memory) ---
LIGHT_RUN = True   # True: fewer CDL counties, lower concurrency, experimental module skips PDF fetch
CDL_MAX_COUNTIES = 400 if LIGHT_RUN else None  # None = all pesticide counties
CDL_MAX_WORKERS = 3  # cap concurrency to avoid rate limits and memory spikes
IPM_SKIP_PDF_IN_LIGHT_RUN = True  # when LIGHT_RUN, use metadata-only experimental scores (no PDF/HTML fetch)


## 1. Load Pesticide Data (County–Year)

Load USGS pesticide estimates for 2016–2019. We keep **one row per county per year** (no aggregation across years) so the joint dataset supports temporal splits and time trends. We offer **multiple aggregation options** and **pesticide categories** by chemical class from Shekhar et al. (2024): [A systematic review of pesticide exposure, associated risks, and long-term human health impacts](https://pmc.ncbi.nlm.nih.gov/articles/PMC11664077/).

**Chemical classes** (linked to respiratory/neurological effects in the literature):
- **Organophosphates** (OP): cholinesterase inhibitors; respiratory, neurotoxic
- **Organochlorines** (OC): persistent; neurotoxic, endocrine disruption
- **Carbamates**: reversible AChE inhibitors; respiratory effects
- **Pyrethroids**: neurotoxic; respiratory distress
- **Triazines** (e.g. atrazine): herbicides; cancer associations
- **Chlorophenoxy** (e.g. 2,4-D): herbicides
- **Other/Unclassified**

In [ ]:
# Pesticide URLs for 2016-2019
PEST_2016_17 = "https://www.sciencebase.gov/catalog/file/get/5e95c12282ce172707f2524e?f=__disk__62%2F83%2Fd3%2F6283d3501f1028b1ccc3976ea2e6de848bc2fef8&allowOpen=true"
PEST_2018 = "https://www.sciencebase.gov/catalog/file/get/6081a706d34e8564d686618e?f=__disk__58%2F6a%2Fed%2F586aed9a844eac0174a0600c8a7293ec4cda0265&allowOpen=true"
PEST_2019 = "https://www.sciencebase.gov/catalog/file/get/6081a924d34e8564d68661a1?f=__disk__08%2F42%2Fcd%2F0842cdac3a7d8b5056645a4dc08d1da96ad4e0b7&allowOpen=true"

# Prefer local copies under ../data/ when present (avoids ScienceBase fetch; useful for CI / offline).
from pathlib import Path
_DATA_DIR = Path("../data").resolve()
_LOCAL_1617 = _DATA_DIR / "pesticide_2016_17.tsv"
_LOCAL_2018 = _DATA_DIR / "pesticide_2018.tsv"
_LOCAL_2019 = _DATA_DIR / "pesticide_2019.tsv"

if _LOCAL_1617.exists() and _LOCAL_2018.exists():
    df_pest = pd.read_csv(_LOCAL_1617, sep="\t")
    df_pest = df_pest[df_pest["YEAR"].isin([2016, 2017])]
    df_pest = pd.concat([df_pest, pd.read_csv(_LOCAL_2018, sep="\t")], ignore_index=True)
    if _LOCAL_2019.exists():
        df_pest = pd.concat([df_pest, pd.read_csv(_LOCAL_2019, sep="\t")], ignore_index=True)
else:
    df_pest = pd.read_csv(PEST_2016_17, sep="\t")
    df_pest = df_pest[df_pest["YEAR"].isin([2016, 2017])]
    df_pest = pd.concat([df_pest, pd.read_csv(PEST_2018, sep="\t"), pd.read_csv(PEST_2019, sep="\t")], ignore_index=True)

# Standardize FIPS
df_pest["STATE_FIPS"] = df_pest["STATE_FIPS_CODE"].astype(int).astype(str).str.zfill(2)
df_pest["COUNTY_FIPS"] = df_pest["COUNTY_FIPS_CODE"].astype(int).astype(str).str.zfill(3)
df_pest["FIPS"] = df_pest["STATE_FIPS"] + df_pest["COUNTY_FIPS"]
# USGS provides EPEST_LOW_KG and EPEST_HIGH_KG; use mean as single estimate (per datasheet)
df_pest["EPEST_MEAN_KG"] = df_pest[["EPEST_LOW_KG", "EPEST_HIGH_KG"]].mean(axis=1)

# --- Pesticide classification by chemical class (Shekhar et al., 2024, PMC11664077) ---
# Compounds matched by substring (case-insensitive). Order matters: more specific first.
CHEMICAL_CLASS_KEYWORDS = {
    "Organochlorine": ["ddt", "aldrin", "endosulfan", "endrin", "chlordane", "lindane", "dieldrin", "heptachlor", "toxaphene", "mirex", "methoxychlor"],
    "Organophosphate": ["chlorpyrifos", "malathion", "parathion", "diazinon", "dichlorvos", "fenthion", "phorate", "ethion", "acephate", "dimethoate", "methamidophos", "terbufos", "phosmet"],
    "Carbamate": ["carbaryl", "carbofuran", "propoxur", "bendiocarb", "aldicarb", "methomyl", "thiodicarb", "oxamyl", "pirimicarb"],
    "Pyrethroid": ["permethrin", "cypermethrin", "deltamethrin", "lambda-cyhalothrin", "cyhalothrin", "bifenthrin", "esfenvalerate", "fenvalerate", "tefluthrin", "pyrethroid"],
    "Triazine": ["atrazine", "propazine", "terbutryn", "cyanazine", "simazine", "terbuthylazine"],
    "Chlorophenoxy": ["2,4-d", "2,4-dichlorophenoxy", "mcpa", "mcpb", "dicamba", "2,4-db"],
    "Anilide": ["alachlor", "pretilachlor", "metolachlor", "acetochlor", "propanil"],
}

def assign_chemical_class(compound: str) -> str:
    """Assign chemical class from compound name (case-insensitive substring match)."""
    c = str(compound).lower()
    for cls, keywords in CHEMICAL_CLASS_KEYWORDS.items():
        if any(kw in c for kw in keywords):
            return cls
    return "Other"

df_pest["chemical_class"] = df_pest["COMPOUND"].apply(assign_chemical_class)

# --- Compute aggregations at county-year level (keep YEAR for temporal analysis) ---
# 1. Total: sum all compounds per county per year. Note that the calculation of total number of pesticides changes in 2019 to include fewer compounds.
pest_total = df_pest.groupby(["FIPS", "YEAR"])["EPEST_MEAN_KG"].sum().reset_index()
pest_total = pest_total.rename(columns={"EPEST_MEAN_KG": "pesticide_total_kg"})

# 2. By class: one column per chemical class (kg) per county-year
pest_by_class = df_pest.groupby(["FIPS", "YEAR", "chemical_class"])["EPEST_MEAN_KG"].sum().reset_index()
pest_class_wide = pest_by_class.pivot(index=["FIPS", "YEAR"], columns="chemical_class", values="EPEST_MEAN_KG").fillna(0).reset_index()
class_cols = [c for c in pest_class_wide.columns if c not in ("FIPS", "YEAR")]
pest_class_wide.columns = ["FIPS", "YEAR"] + [f"pesticide_{c.lower()}_kg" for c in class_cols]

# 3. Respiratory-relevant: sum only OP, Carbamate, Pyrethroid (strongest respiratory links) per county-year
resp_classes = ["Organophosphate", "Carbamate", "Pyrethroid"]
df_resp = df_pest[df_pest["chemical_class"].isin(resp_classes)]
pest_resp = df_resp.groupby(["FIPS", "YEAR"])["EPEST_MEAN_KG"].sum().reset_index()
pest_resp = pest_resp.rename(columns={"EPEST_MEAN_KG": "pesticide_respiratory_kg"})

# 4. By compound: one column per compound (kg) per county-year
def sanitize_compound_col(name):
    """Convert compound name to valid column name: pesticide_<sanitized>_kg"""
    s = re.sub(r"[^a-zA-Z0-9]", "_", str(name).strip()).lower()
    s = re.sub(r"_+", "_", s).strip("_")  # collapse multiple underscores
    return f"pesticide_{s}_kg" if s else "pesticide_unknown_kg"

pest_by_compound = df_pest.groupby(["FIPS", "YEAR", "COMPOUND"])["EPEST_MEAN_KG"].sum().reset_index()
pest_compound_wide = pest_by_compound.pivot(index=["FIPS", "YEAR"], columns="COMPOUND", values="EPEST_MEAN_KG").fillna(0).reset_index()
# Sanitize compound names for column headers; handle duplicates
seen = {}
new_cols = []
for c in pest_compound_wide.columns:
    if c in ("FIPS", "YEAR"):
        new_cols.append(c)
        continue
    name = sanitize_compound_col(c)
    seen[name] = seen.get(name, 0) + 1
    if seen[name] > 1:
        name = name.replace("_kg", f"_{seen[name]}_kg")
    new_cols.append(name)
pest_compound_wide.columns = new_cols

# Merge all into pest_county (one row per county-year)
pest_county = pest_total.merge(pest_class_wide, on=["FIPS", "YEAR"], how="left").merge(pest_resp, on=["FIPS", "YEAR"], how="left")
pest_county = pest_county.merge(pest_compound_wide, on=["FIPS", "YEAR"], how="left")
pest_county["pesticide_respiratory_kg"] = pest_county["pesticide_respiratory_kg"].fillna(0)

print("Pesticide (county-year):", pest_county.shape)
print("  - Total + respiratory +", pest_class_wide.shape[1] - 2, "class cols +", pest_compound_wide.shape[1] - 2, "compound cols (one row per county-year)")
print("Chemical class distribution:", df_pest["chemical_class"].value_counts())
pest_county.head()


# ultimately it can be ok to to have a large number of features from the chemical compounds and we can observe if there are features downwighted seariously by the model itself. 

## 2. Load Census Demographics (County-level)

ACS 5-year (2019) for population, median age, income; **race percentages** (pct_white, pct_black, pct_asian, pct_hispanic). **Rural/urban:** NCHS 6-level classification (1–6) from [CDC NCHS Urban-Rural](https://www.cdc.gov/nchs/data-analysis-tools/urban-rural.html), merged on county FIPS (state FIPS + county FIPS).

In [ ]:
CENSUS_BASE = "https://api.census.gov/data"
VARS = ["NAME", "B01003_001E", "B01002_001E", "B19013_001E", "B03002_001E", "B03002_003E", "B03002_004E", "B03002_006E", "B03002_012E"]

resp = requests.get(f"{CENSUS_BASE}/2019/acs/acs5", params={"get": ",".join(VARS), "for": "county:*", "in": "state:*"})
resp.raise_for_status()
data = resp.json()
census = pd.DataFrame(data[1:], columns=data[0])

census["FIPS"] = census["state"].astype(str).str.zfill(2) + census["county"].astype(str).str.zfill(3)
for c in ["B01003_001E", "B01002_001E", "B19013_001E", "B03002_001E", "B03002_003E", "B03002_004E", "B03002_006E", "B03002_012E"]:
    census[c] = pd.to_numeric(census[c], errors="coerce")

census = census.rename(columns={"B01003_001E": "population", "B01002_001E": "median_age", "B19013_001E": "median_income", "B03002_001E": "pop_race"})
# Race percentages (demographic confounders for respiratory health)
for race, col in [("pct_white", "B03002_003E"), ("pct_black", "B03002_004E"), ("pct_asian", "B03002_006E"), ("pct_hispanic", "B03002_012E")]:
    census[race] = 100 * census[col] / census["pop_race"].replace(0, np.nan)

# Rural/urban: NCHS 6-level classification (1–6), county-level; merge on FIPS = state FIPS (2) + county FIPS (3)
# https://www.cdc.gov/nchs/data-analysis-tools/urban-rural.html  (2023 scheme: CODE2023)
NCHS_URBAN_RURAL_URL = "https://www.cdc.gov/nchs/data/data-analysis/NCHSurb-rural-codes.csv"
try:
    r_nchs = requests.get(NCHS_URBAN_RURAL_URL, timeout=30)
    r_nchs.raise_for_status()
    nchs = pd.read_csv(StringIO(r_nchs.text), dtype=str)
    nchs.columns = [c.strip() for c in nchs.columns]
    # Per CDC file docs: STFIPS, CTYFIPS; full FIPS = zfill(2) + zfill(3)
    state_col = "STFIPS" if "STFIPS" in nchs.columns else next((c for c in nchs.columns if "state" in c.lower() and "fips" in c.lower()), nchs.columns[0])
    county_col = "CTYFIPS" if "CTYFIPS" in nchs.columns else next((c for c in nchs.columns if "county" in c.lower() and "fips" in c.lower()), nchs.columns[1])
    nchs["FIPS"] = nchs[state_col].astype(str).str.zfill(2) + nchs[county_col].astype(str).str.zfill(3)
    code_col = "CODE2023" if "CODE2023" in nchs.columns else next((c for c in nchs.columns if "2023" in c or "code" in c.lower()), None)
    if code_col is None:
        code_col = [c for c in nchs.columns if c not in (state_col, county_col, "FIPS", "CTYNAME", "ST_ABBREV")][0]
    nchs["nchs_urban_rural"] = pd.to_numeric(nchs[code_col], errors="coerce")
    nchs = nchs[["FIPS", "nchs_urban_rural"]].dropna(subset=["nchs_urban_rural"])
    census = census.merge(nchs, on="FIPS", how="left")
    print("Rural/urban: added nchs_urban_rural (1–6) from CDC NCHS; 1=Large central metro, 2=Large fringe, 3=Medium, 4=Small metro, 5=Micropolitan, 6=Noncore.")
except Exception as e:
    print("NCHS urban-rural failed:", e)
    nchs_path = os.path.join(os.path.dirname(os.getcwd()), "data", "NCHSurb-rural-codes.csv")
    if os.path.isfile(nchs_path):
        nchs = pd.read_csv(nchs_path, dtype=str)
        nchs.columns = [c.strip() for c in nchs.columns]
        state_col = "STFIPS" if "STFIPS" in nchs.columns else [c for c in nchs.columns if "state" in c.lower() and "fips" in c.lower()][0]
        county_col = "CTYFIPS" if "CTYFIPS" in nchs.columns else [c for c in nchs.columns if "county" in c.lower() and "fips" in c.lower()][0]
        code_col = "CODE2023" if "CODE2023" in nchs.columns else [c for c in nchs.columns if "code" in c.lower()][0]
        nchs["FIPS"] = nchs[state_col].astype(str).str.zfill(2) + nchs[county_col].astype(str).str.zfill(3)
        nchs["nchs_urban_rural"] = pd.to_numeric(nchs[code_col], errors="coerce")
        census = census.merge(nchs[["FIPS", "nchs_urban_rural"]].dropna(subset=["nchs_urban_rural"]), on="FIPS", how="left")
        print("Loaded NCHS urban-rural from local data/NCHSurb-rural-codes.csv")
    else:
        census["nchs_urban_rural"] = np.nan

print("Census (county-level):", census.shape)
census.head()

## 3. Load Cropland Data (County-level)

The CropScape API returns **acreage by land-cover category** (e.g., Corn, Soybeans, Cotton, Forest, Developed). We extract multiple factors beyond total acres.

**Fetch strategy:** We fetch CDL for **all** pesticide counties using **parallel requests**. A short **probe** (15 test requests) runs first to find a safe concurrency level — the API doesn't publish rate limits, so we try 10 workers, then 5, 2, or 1 if we hit 429/503 errors.

**Cropland factors extracted** (likely relevant for pesticide/health modeling):

| Factor | Rationale |
|--------|------------|
| `cropland_acres` | Total cultivated cropland — baseline agricultural intensity |
| `pct_cropland` | Share of county in crops — intensity relative to land area |
| `corn_acres`, `soybean_acres`, `cotton_acres` | Major row crops — different pesticide profiles (e.g., atrazine on corn) |
| `fruit_veg_acres` | Orchards, vineyards, vegetables — often higher pesticide use per acre |
| `hay_acres` | Alfalfa/hay — moderate use |
| `cropland_diversity` | Number of crop types — monoculture vs. mixed farming |

In [ ]:
CDL_API = "https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLStat"

def get_county_cdl(year: int, fips: str) -> pd.DataFrame:
    params = {"year": year, "fips": fips, "format": "csv"}
    r = requests.get(CDL_API, params=params)
    r.raise_for_status()
    root = ET.fromstring(r.text)
    ns = {"ns": "http://cropscape.csiss.gmu.edu/CDLService/"}
    url_elem = root.find(".//ns:returnURL", ns) or root.find(".//returnURL")
    data_url = url_elem.text.strip() if url_elem is not None else f"https://nassgeodata.gmu.edu/webservice/nass_data_cache/byfips/CDL_{year}_{fips}.csv"
    r2 = requests.get(data_url)
    r2.raise_for_status()
    return pd.read_csv(StringIO(r2.text))

# Fetch cropland for pesticide counties (parallel requests; use subset if LIGHT_RUN)
all_fips = pest_county["FIPS"].astype(str).str.zfill(5).unique().tolist()
if CDL_MAX_COUNTIES is not None:
    all_fips = all_fips[: int(CDL_MAX_COUNTIES)]
    print("CDL: using first", len(all_fips), "counties (LIGHT_RUN or CDL_MAX_COUNTIES).")

# Probe: discover safe concurrency (API doesn't publish rate limits)
# Try a small batch; if we get 429/503, reduce workers and retry
def probe_safe_workers(max_workers_start=10, probe_size=15):
    probe_fips = all_fips[:probe_size]
    for n in [max_workers_start, 5, 2, 1]:
        errors = 0
        with ThreadPoolExecutor(max_workers=n) as ex:
            futures = [ex.submit(get_county_cdl, 2019, f) for f in probe_fips]
            for f in as_completed(futures):
                try:
                    f.result()
                except requests.exceptions.HTTPError as e:
                    if e.response.status_code in (429, 503):
                        errors += 1
                except Exception:
                    errors += 1
        if errors == 0:
            print(f"Probe OK with {n} worker(s). Using N_WORKERS={n}.")
            return n
        print(f"Probe with {n} worker(s): {errors} rate-limit/server errors. Trying fewer...")
        time.sleep(2)
    return 1

N_WORKERS = probe_safe_workers(max_workers_start=10, probe_size=15)

CROP_GROUPS = {
    "corn_acres": ["corn", "sweet corn", "maize"],
    "soybean_acres": ["soybean", "soybeans", "soy"],
    "cotton_acres": ["cotton"],
    "wheat_acres": ["wheat", "durum", "spring wheat", "winter wheat"],
    "hay_acres": ["hay", "alfalfa"],
    "fruit_veg_acres": [
        "orchard", "orchards", "vineyard", "vineyards", "fruit", "fruits", "vegetable", "vegetables",
        "grape", "grapes", "apple", "apples", "citrus", "berry", "berries", "tomato", "tomatoes",
        "peach", "peaches", "almond", "almonds", "potato", "potatoes", "melon", "melons", "lettuce",
        "onion", "onions", "carrot", "carrots", "broccoli", "pepper", "peppers", "strawberry", "strawberries"
    ],
    "rice_acres": ["rice"],
    "sorghum_acres": ["sorghum"],
}
NON_CROP = ["developed", "open water", "water", "forest", "wetland", "barren", "shrubland", "grassland", "pasture"]

CANONICAL_CROP_KEYWORDS = {
    "corn": ["sweet corn", "maize", "corn"],
    "soybean": ["soybeans", "soybean", "soy"],
    "cotton": ["cotton"],
    "wheat": ["durum", "spring wheat", "winter wheat", "wheat"],
    "hay": ["alfalfa", "hay"],
    "fruit_veg": [
        "strawberries", "strawberry", "vegetables", "vegetable", "vineyards", "vineyard", "orchards", "orchard",
        "potatoes", "potato", "tomatoes", "tomato", "berries", "berry", "apples", "apple", "almonds", "almond",
        "peaches", "peach", "grapes", "grape", "citrus", "melons", "melon", "lettuce", "onions", "onion",
        "carrots", "carrot", "broccoli", "peppers", "pepper", "fruit", "fruits"
    ],
    "rice": ["rice"],
    "sorghum": ["sorghum"],
}
CROP_FAMILY_MAP = {
    "corn": "field_crop",
    "soybean": "field_crop",
    "cotton": "field_crop",
    "wheat": "field_crop",
    "rice": "field_crop",
    "sorghum": "field_crop",
    "hay": "forage",
    "fruit_veg": "specialty_crop",
    "other_crop": "other_crop",
}

_SORTED_CROP_KEYWORDS = sorted(
    [(keyword, crop) for crop, keywords in CANONICAL_CROP_KEYWORDS.items() for keyword in keywords],
    key=lambda kv: len(kv[0]),
    reverse=True,
)

def normalize_crop_label(name: str) -> str:
    text = re.sub(r"[^a-z0-9]+", " ", str(name or "").lower()).strip()
    for keyword, crop in _SORTED_CROP_KEYWORDS:
        if re.search(rf"\b{re.escape(keyword)}\b", text):
            return crop
    return "other_crop"

def crop_family(crop: str) -> str:
    return CROP_FAMILY_MAP.get(str(crop), "other_crop")

def cdl_to_long(df_c: pd.DataFrame, fips: str, year: int) -> list:
    """Convert CDL stat response to long format (FIPS, year, crop, crop_family, acres) for county_crop_year_ag."""
    acre_col = next((c for c in ["Acres", "ACRES", "acres"] if c in df_c.columns), None)
    if acre_col is None:
        cand = [c for c in df_c.columns if "acre" in c.lower() or "area" in c.lower()]
        acre_col = cand[0] if cand else df_c.select_dtypes(include=[np.number]).columns[0]
    name_col = next((c for c in ["Category", "Crop", "Class_Name", "NAME"] if c in df_c.columns), df_c.columns[0])
    df_c = df_c.copy()
    df_c["_name"] = df_c[name_col].astype(str).str.lower()
    non_crop_mask = df_c["_name"].str.contains("|".join(NON_CROP), case=False, na=False)
    df_c = df_c.loc[~non_crop_mask]
    rows = []
    for _, r in df_c.iterrows():
        name, ac = r["_name"], r[acre_col]
        if ac <= 0:
            continue
        crop = normalize_crop_label(name)
        rows.append({"FIPS": fips, "year": year, "crop": crop, "crop_family": crop_family(crop), "acres": ac})
    return rows

def extract_crop_features(df_c: pd.DataFrame) -> dict:
    """Extract cropland features from CDL stat response."""
    acre_col = next((c for c in ["Acres", "ACRES", "acres"] if c in df_c.columns), None)
    if acre_col is None:
        cand = [c for c in df_c.columns if "acre" in c.lower() or "area" in c.lower()]
        acre_col = cand[0] if cand else df_c.select_dtypes(include=[np.number]).columns[0]
    name_col = next((c for c in ["Category", "Crop", "Class_Name", "NAME"] if c in df_c.columns), df_c.columns[0])
    df_c = df_c.copy()
    df_c["_name"] = df_c[name_col].astype(str).str.lower()
    total_area = df_c[acre_col].sum()
    crop_mask = ~df_c["_name"].str.contains("|".join(NON_CROP), case=False, na=False)
    cropland_total = df_c.loc[crop_mask, acre_col].sum()
    out = {"cropland_acres": cropland_total, "total_area_acres": total_area, "pct_cropland": 100 * cropland_total / total_area if total_area > 0 else 0}
    for feat, keywords in CROP_GROUPS.items():
        mask = df_c["_name"].str.contains("|".join(re.escape(k) for k in keywords), case=False, na=False)
        out[feat] = df_c.loc[mask, acre_col].sum()
    out["cropland_diversity"] = crop_mask.sum()
    return out

CDL_YEAR = 2019

def fetch_one(fips):
    try:
        df_c = get_county_cdl(CDL_YEAR, fips)
        feats = extract_crop_features(df_c)
        feats["FIPS"] = fips
        long_rows = cdl_to_long(df_c, fips, CDL_YEAR)
        return (feats, long_rows)
    except Exception:
        return None

cropland_rows = []
county_crop_year_ag_rows = []
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = {ex.submit(fetch_one, f): f for f in all_fips}
    for i, fut in enumerate(as_completed(futures), 1):
        out = fut.result()
        if out is not None:
            feats, long_rows = out
            cropland_rows.append(feats)
            county_crop_year_ag_rows.extend(long_rows)
        if i % 250 == 0:
            print(f"Fetched {i}/{len(all_fips)} counties")

cropland = pd.DataFrame(cropland_rows)
print("Cropland feature rows:", cropland.shape)
print("Columns:", list(cropland.columns))

# Build county_crop_year_ag: long table (FIPS, year, crop, crop_family, acres, share_county_crop_acres)
# Repeat CDL_YEAR composition for all analysis years 2016-2019 so joint join has one row per county-year
analysis_years = sorted(pest_county["YEAR"].dropna().unique().astype(int).tolist())
ccy_ag = pd.DataFrame(county_crop_year_ag_rows)
if len(ccy_ag) > 0:
    ccy_list = []
    for yr in analysis_years:
        ccy_yr = ccy_ag.copy()
        ccy_yr["year"] = yr
        ccy_list.append(ccy_yr)
    ccy_ag = pd.concat(ccy_list, ignore_index=True)
    tot_acres = ccy_ag.groupby(["FIPS", "year"])["acres"].transform("sum")
    ccy_ag["share_county_crop_acres"] = np.where(tot_acres > 0, ccy_ag["acres"] / tot_acres, np.nan)
    ccy_ag["crop_value"] = np.nan
    ccy_ag["share_county_crop_value"] = np.nan
    county_crop_year_ag = ccy_ag
    print("County-crop-year (county_crop_year_ag):", county_crop_year_ag.shape)
    if "crop" in county_crop_year_ag.columns:
        print("  Crop mix:", county_crop_year_ag["crop"].value_counts().head(12).to_dict())
else:
    county_crop_year_ag = pd.DataFrame(columns=["FIPS", "year", "crop", "crop_family", "acres", "share_county_crop_acres", "crop_value", "share_county_crop_value"])

cropland.head()


In [ ]:
cropland.describe()

## 3b. Optional experimental bridge (archived in ipm-testing)

IPM sources (Crop Profiles, PMSPs) are **crop × state/region × document**, not county-level. This section implements **section-aware scoring** inline (lexicons for monitoring, nonchemical, chemical, decision support, dependency, resistance management; weak metadata priors; optional PDF fallback). We integrate by:

1. **county_crop_year_ag** — crop composition (acres, shares) per county-year (built above).
2. **crop_geo_doc_ipm** — IPM document-derived scores per crop × state × document year (National IPM Database API; section-aware subscales + ipm_breadth_index, chemical_reliance_index).
3. **county_crop_year_ipm** — aggregate to one row per (crop, state_fips), then join county crops to crop–state IPM on crop + state.
4. **county_year_collapsed** — roll up to county-year with **acreage-weighted** (primary) and **value-weighted** (sensitivity) IPM breadth and chemical reliance.

Interpretation: *"The IPM characteristics of the crop mix grown in this county"* — not county-level adoption.

**Code:** `ipm-testing/ipm_pipeline.py`. **Debug / API checks / inspect joint experimental columns:** `ipm-testing/ipm_data_load_troubleshoot.ipynb`.

In [ ]:
# --- archived experimental bridge: logic lives in `ipm-testing/ipm_pipeline.py` (see also `ipm-testing/ipm_data_load_troubleshoot.ipynb`) ---
import numpy as np
import pandas as pd

import os, sys
sys.path.append(os.path.join(os.getcwd(), "ipm-testing"))
from ipm_pipeline import (
    ANALYSIS_YEARS_IPM,
    build_crop_geo_doc_ipm,
    aggregate_crop_geo_doc_ipm_to_crop_state,
    match_county_crop_year_ipm,
    collapse_county_year,
    _crop_family,
)

_use_pdf = True
try:
    _use_pdf = not (LIGHT_RUN and IPM_SKIP_PDF_IN_LIGHT_RUN)
except NameError:
    pass

try:
    crop_geo_doc_ipm = build_crop_geo_doc_ipm(use_pdf_fallback=_use_pdf)
    print("crop_geo_doc_ipm (section-aware, inline):", crop_geo_doc_ipm.shape)
    if len(crop_geo_doc_ipm) > 0 and "text_source" in crop_geo_doc_ipm.columns:
        print("  Text source:", crop_geo_doc_ipm["text_source"].value_counts().to_dict())
    if len(crop_geo_doc_ipm) > 0 and "crop" in crop_geo_doc_ipm.columns:
        print("  Crop coverage:", crop_geo_doc_ipm["crop"].value_counts().head(12).to_dict())
except Exception as e:
    print("build_crop_geo_doc_ipm failed:", e)
    crop_geo_doc_ipm = pd.DataFrame(columns=["crop", "crop_family", "state_fips", "document_year", "document_type", "source_id", "url", "ipm_breadth_index", "chemical_reliance_index", "monitoring_score", "nonchemical_score"])

crop_state_ipm = aggregate_crop_geo_doc_ipm_to_crop_state(crop_geo_doc_ipm, analysis_years=ANALYSIS_YEARS_IPM)

ccy = county_crop_year_ag.copy()
ccy["state_fips"] = ccy["FIPS"].astype(str).str.zfill(5).str[:2]
ccy["crop"] = ccy["crop"].astype(str)
if "crop_family" not in ccy.columns:
    ccy["crop_family"] = ccy["crop"].map(_crop_family)
if "year" in ccy.columns:
    ccy["year"] = pd.to_numeric(ccy["year"], errors="coerce")
    ccy = ccy[ccy["year"].notna()].copy()
    ccy["year"] = ccy["year"].astype(int)
    ccy = ccy[ccy["year"].isin(ANALYSIS_YEARS_IPM)].copy()
if len(crop_state_ipm) > 0:
    crop_state_ipm = crop_state_ipm.copy()
    crop_state_ipm["crop"] = crop_state_ipm["crop"].astype(str)
    crop_state_ipm["crop_family"] = crop_state_ipm["crop_family"].astype(str)
    crop_state_ipm["state_fips"] = crop_state_ipm["state_fips"].astype(str)
    crop_state_ipm["year"] = pd.to_numeric(crop_state_ipm["year"], errors="coerce").astype(int)
    print("crop_state_ipm (for merge):", crop_state_ipm.shape, "| crops:", sorted(crop_state_ipm["crop"].unique().tolist())[:15], "| years:", sorted(crop_state_ipm["year"].dropna().unique().tolist()))
if len(ccy) > 0 and len(crop_state_ipm) > 0:
    county_crop_year_ipm = match_county_crop_year_ipm(ccy, crop_state_ipm)
    n_match = county_crop_year_ipm["ipm_breadth_index"].notna().sum()
    print("county_crop_year_ipm:", county_crop_year_ipm.shape, "| rows with experimental match:", n_match, ("%.1f%%" % (100 * n_match / len(county_crop_year_ipm) if len(county_crop_year_ipm) else 0)))
    if "ipm_match_tier" in county_crop_year_ipm.columns:
        print("  Match tiers:", county_crop_year_ipm["ipm_match_tier"].fillna("unmatched").value_counts().to_dict())
else:
    county_crop_year_ipm = ccy.copy()
    for col in ["ipm_breadth_index", "chemical_reliance_index", "ipm_breadth_index_rescaled", "chemical_reliance_index_rescaled", "mean_text_quality", "mean_geo_confidence", "weighted_doc_age", "n_docs", "ipm_match_tier", "ipm_source_crop", "ipm_source_crop_family", "ipm_source_state_fips"]:
        county_crop_year_ipm[col] = np.nan
if len(ccy) == 0:
    county_crop_year_ipm = pd.DataFrame()

county_year_collapsed = collapse_county_year(county_crop_year_ipm)
if len(county_year_collapsed) > 0:
    print("County-year collapsed:", county_year_collapsed.shape)
    print("  Columns:", list(county_year_collapsed.columns))
    has_any_value = county_crop_year_ag["crop_value"].notna().any() if "crop_value" in county_crop_year_ag.columns else False
    if not has_any_value:
        print("  Note: ipm_breadth_value / chemical_reliance_value / total_ag_value are all NaN (crop_value not populated; add NASS for value-weighted experimental scoring).")
    if "ipm_breadth_index" in county_crop_year_ipm.columns:
        n_ccy = len(county_crop_year_ipm)
        n_with_ipm = county_crop_year_ipm["ipm_breadth_index"].notna().sum()
        print("  County-crop-year rows with non-null experimental score (acre): %.0f / %.0f (%.1f%%)" % (n_with_ipm, n_ccy, 100 * n_with_ipm / n_ccy if n_ccy else 0))
    if "ipm_breadth_acre" in county_year_collapsed.columns:
        n_cy = len(county_year_collapsed)
        n_cy_with_ipm = county_year_collapsed["ipm_breadth_acre"].notna().sum()
        n_cy_nonzero = (county_year_collapsed["ipm_breadth_acre"].notna() & (county_year_collapsed["ipm_breadth_acre"] > 0)).sum()
        n_cy_with_coverage = (county_year_collapsed["ipm_doc_coverage_share"] > 0).sum() if "ipm_doc_coverage_share" in county_year_collapsed.columns else 0
        print("  County-year rows with non-null ipm_breadth_acre: %d / %d (%.1f%%)" % (n_cy_with_ipm, n_cy, 100 * n_cy_with_ipm / n_cy if n_cy else 0))
        print("  County-year rows with ipm_breadth_acre > 0: %d" % n_cy_nonzero)
        print("  County-year rows with any experimental document coverage: %d / %d (%.1f%%)" % (n_cy_with_coverage, n_cy, 100 * n_cy_with_coverage / n_cy if n_cy else 0))
        if "ipm_primary_match_tier" in county_year_collapsed.columns:
            print("  Primary county-year match tiers:", county_year_collapsed["ipm_primary_match_tier"].fillna("unmatched").value_counts().to_dict())
else:
    if len(county_crop_year_ag) > 0:
        ccy_ag = county_crop_year_ag.copy()
        for c in ["ipm_breadth_index", "chemical_reliance_index"]:
            ccy_ag[c] = np.nan
        county_year_collapsed = collapse_county_year(ccy_ag)
        print("County-year collapsed (no experimental scores yet):", county_year_collapsed.shape)


In [ ]:
crop_geo_doc_ipm



## 4. Load CDC PLACES (county-year, 2018 and 2019 only)

Uses **CDCPLACES R package** via `rpy2`. County-level for **2018 and 2019 only** (release 2020 = 2018 BRFSS, 2021 = 2019 BRFSS). One row per county per year.

"Merge to joint on FIPS and YEAR; 2016/2017 rows get NaN for PLACES.\n",

**Measures:** CASTHMA, COPD, CSMOKING, OBESITY, DIABETES.

In [ ]:
# CDC PLACES: county-level health outcomes for 2018 and 2019 (release 2020 = 2018 BRFSS, 2021 = 2019 BRFSS)
# Note: 2020 API has no locationid; 2021 has locationid (FIPS). We fetch 2021 first to build (stateabbr, locationname)->FIPS lookup for 2020.
PLACES_MEASURES = ["CASTHMA", "COPD", "CSMOKING", "OBESITY", "DIABETES"]
PLACES_RELEASE_TO_YEAR = {"2021": 2019, "2020": 2018}  # 2021 first so we have FIPS lookup for 2020

try:
    cdcplaces = importr("CDCPLACES")
    r.assign("measures_r", StrVector(PLACES_MEASURES))
    tmp_csv = os.path.join(tempfile.gettempdir(), "places_county_r.csv")
    r.assign("tmp_path", tmp_csv)
    places_list = []
    name_to_fips = None  # (stateabbr, locationname) -> FIPS from 2021 data
    for release_str, analysis_year in PLACES_RELEASE_TO_YEAR.items():
        r.assign("release_r", release_str)
        r('places_county <- get_places(geography = "county", state = NULL, measure = measures_r, release = release_r, geometry = FALSE)')
        r('''
        atomic <- sapply(places_county, is.atomic)
        places_county <- places_county[, atomic, drop = FALSE]
        write.csv(places_county, tmp_path, row.names = FALSE)
        ''')
        county_df = pd.read_csv(tmp_csv)
        county_df.columns = [c.strip().lower().replace(" ", "_") for c in county_df.columns]
        measure_col = "measureid" if "measureid" in county_df.columns else [c for c in county_df.columns if "measure" in c][0]
        value_col = "data_value" if "data_value" in county_df.columns else [c for c in county_df.columns if "value" in c and "conf" not in c][0]
        id_col = next((c for c in ["locationid", "location_id", "geolocation", "countyfips", "fips"] if c in county_df.columns), None)
        has_locationid = id_col is not None
        if not has_locationid:
            id_col = [c for c in county_df.columns if "location" in c or "fips" in c or c == "geoid"][0]
        if has_locationid:
            county_wide = county_df.pivot_table(index=id_col, columns=measure_col, values=value_col, aggfunc="first").reset_index()
            county_wide["FIPS"] = county_wide[id_col].astype(str).str.replace(r"\\D", "", regex=True).str.zfill(5)
            if name_to_fips is None and "stateabbr" in county_df.columns and "locationname" in county_df.columns:
                name_to_fips = county_df[["stateabbr", "locationname", id_col]].drop_duplicates()
                name_to_fips["FIPS"] = name_to_fips[id_col].astype(str).str.replace(r"\\D", "", regex=True).str.zfill(5)
                name_to_fips = name_to_fips[["stateabbr", "locationname", "FIPS"]].drop_duplicates()
        else:
            if "stateabbr" not in county_df.columns or id_col not in county_df.columns:
                continue
            county_wide = county_df.pivot_table(index=["stateabbr", id_col], columns=measure_col, values=value_col, aggfunc="first").reset_index()
            county_wide = county_wide.rename(columns={id_col: "locationname"})
            if name_to_fips is not None:
                county_wide = county_wide.merge(name_to_fips, on=["stateabbr", "locationname"], how="left")
                county_wide = county_wide.dropna(subset=["FIPS"])
                county_wide = county_wide.drop(columns=["stateabbr", "locationname"], errors="ignore")
            else:
                county_wide["FIPS"] = ""
        keep_cols = ["FIPS"] + [c for c in PLACES_MEASURES if c in county_wide.columns]
        df = county_wide[keep_cols].copy()
        df["YEAR"] = analysis_year
        places_list.append(df)
    try:
        os.remove(tmp_csv)
    except OSError:
        pass
    places = pd.concat(places_list, ignore_index=True)
    print("Loaded CDC PLACES (county-year): 2018 and 2019 only (releases 2020, 2021).")
except Exception as e:
    print(f"CDCPLACES failed: {e}")
    places = pd.DataFrame(columns=["FIPS", "YEAR", "CASTHMA", "COPD", "CSMOKING", "OBESITY", "DIABETES"])

# Normalize so merge on FIPS+YEAR matches: FIPS as 5-char string, YEAR as int
if len(places) > 0 and "FIPS" in places.columns and "YEAR" in places.columns:
    places = places.copy()
    places["FIPS"] = places["FIPS"].astype(str).str.replace(r"\\D", "", regex=True).str.zfill(5)
    places = places[places["FIPS"].str.len() == 5].copy()  # drop invalid FIPS
    places["YEAR"] = pd.to_numeric(places["YEAR"], errors="coerce")
    places = places[places["YEAR"].notna()].copy()
    places["YEAR"] = places["YEAR"].astype(int)
    places = places.drop_duplicates(subset=["FIPS", "YEAR"])
    print("PLACES years present:", sorted(places["YEAR"].unique().tolist()))

print("PLACES (county-year):", places.shape)
places.head()

In [ ]:
places.tail(1000)

## 5. Join All Datasets (County–Year on FIPS)

In [ ]:
# Focus on 2018 and 2019 only (years with CDC PLACES data)
ANALYSIS_YEARS_JOINT = [2018, 2019]
joint = pest_county[pest_county["YEAR"].isin(ANALYSIS_YEARS_JOINT)].copy()
joint["YEAR"] = joint["YEAR"].astype(int)
joint["FIPS"] = joint["FIPS"].astype(str).str.replace(r"\\D", "", regex=True).str.zfill(5)

# Merge census and cropland on FIPS (county-level; repeated for each year); merge PLACES on FIPS + YEAR
census_cols = ["FIPS", "NAME", "population", "median_age", "median_income"]
census_cols += [c for c in ["pct_white", "pct_black", "pct_asian", "pct_hispanic", "nchs_urban_rural"] if c in census.columns]
census_sub = census[census_cols].copy()
joint = joint.merge(census_sub, on="FIPS", how="left", suffixes=("", "_census"))

# Merge cropland (left join)
joint = joint.merge(cropland, on="FIPS", how="left")

# Merge PLACES (on FIPS and YEAR; both must be same type for match)
if "YEAR" in places.columns and len(places) > 0:
    joint = joint.merge(places, on=["FIPS", "YEAR"], how="left")
    if "CASTHMA" in joint.columns:
        print("PLACES merge by year - rows with non-null CASTHMA:", joint.groupby("YEAR")["CASTHMA"].apply(lambda x: x.notna().sum()).to_dict())
else:
    joint = joint.merge(places, on="FIPS", how="left") if len(places) > 0 else joint
joint["state_fips"] = joint["FIPS"].astype(str).str.zfill(5).str[:2]

# Merge county-year collapsed crop/IPM indices (acreage- and value-weighted; primary crop representation for archived experimental scoring)
if len(county_year_collapsed) > 0:
    cyc = county_year_collapsed.rename(columns={"year": "YEAR"})
    joint = joint.merge(cyc, on=["FIPS", "YEAR"], how="left")
    print("Merged county_year_collapsed (ipm_breadth_acre, chemical_reliance_acre, county_crop_concentration, specialty_crop_share, etc.)")

print("Joint dataset shape (county-year rows):", joint.shape)
print("  Years:", sorted(joint["YEAR"].dropna().unique().tolist()))
print("\nColumn coverage:")
print(joint.notna().sum())
joint.head(10)

## 6. Save joint dataset

Write the joined table to CSV for modeling and for **`joint_eda.ipynb`**.


In [ ]:
out_path = "../data/joint_county_year_2018_2019.csv"
joint.to_csv(out_path, index=False)
print("Saved:", out_path, "(", len(joint), "rows )")
